# 📝 Datathon Template — TEXT DATA (NLP, Text Classification)

| # | Step | কি করব |
|---|------|--------|
| 0 | Setup & Config | path, text column, target, language (en/bn) |
| 1 | Load data | |
| 2 | Cleaning | missing / empty text, duplicate, conflicting label |
| 3 | EDA | class balance, length, word count, top n-grams, wordcloud, special tokens, train vs test |
| 4 | Text preprocessing | lowercase, URL/HTML/emoji, punctuation, stopwords, stemming/lemmatization (English + বাংলা) |
| 5 | Meta features | length, punctuation, uppercase ratio ... |
| 6 | Vectorization | Bag-of-Words, TF-IDF (word + char n-gram) |
| 7 | Model Zoo | LogReg, LinearSVC, NB, SGD, Ridge, LightGBM ... CV compare |
| 8 | Tuning | GridSearchCV |
| 9 | OOF + Evaluation | report, confusion matrix, error analysis |
| 10 | Interpretation | প্রতিটা class এর top words |
| 11 | Sentence embeddings | LSA (SVD) visualization, sentence-transformers + LogReg |
| 12 | Transformer fine-tuning | BERT / DeBERTa / XLM-R / BanglaBERT (GPU) |
| 13 | Ensemble + Submission | |

> কোনো data না দিলে notebook নিজেই একটা **product review sentiment** demo dataset বানাবে (URL, HTML, emoji, duplicate সহ)।

## ☁️ Google Colab Setup (সবার আগে এটা চালাও)

**GPU চালু করা:** `Runtime → Change runtime type → T4 GPU → Save` (image / transformer এর জন্য জরুরি)

**Data আনার ৪টা উপায় — নিচের `DATA_SOURCE` বদলাও:**
| `DATA_SOURCE` | কিভাবে |
|---|---|
| `"demo"` | কিছু লাগবে না — notebook নিজেই practice data বানাবে |
| `"upload"` | Cell চালালে **Choose Files** button আসবে → `train.csv`, `test.csv`, `sample_submission.csv` (বা একটা `.zip`) select করো |
| `"drive"` | আগে Google Drive এ data রাখো (যেমন `MyDrive/datathon/nlp/`) → `DRIVE_DATA_DIR` ঠিক করো → permission দাও |
| `"kaggle"` | Kaggle → Settings → **Create New Token** → `kaggle.json` download → cell চালালে upload করো; competition page এ আগে **rules accept** করতে হবে |

সব data শেষে `data/nlp` folder এ আসবে, `.zip` থাকলে auto unzip হবে — তাই নিচের CONFIG এ path বদলাতে হবে না
(শুধু file / column এর নাম মিলিয়ে নিও)।

> ⚠️ Colab বন্ধ হলে `/content` এর সব file মুছে যায় — submission আর model notebook এর শেষ cell দিয়ে download / Drive এ save করো।

In [ ]:
import sys, os, shutil, zipfile, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# ---------- 1) যে library Colab এ নেই সেগুলো install ----------
NEED = {"seaborn": "seaborn", "wordcloud": "wordcloud", "nltk": "nltk", "lightgbm": "lightgbm", "transformers": "transformers", "sentence_transformers": "sentence-transformers", "sentencepiece": "sentencepiece"}          # {import_name: pip_name}
missing = [pip for mod, pip in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

# ---------- 2) Data ----------
DATA_SOURCE = "demo"                                  # "demo" | "upload" | "drive" | "kaggle"
DRIVE_DATA_DIR = "/content/drive/MyDrive/datathon/nlp"
KAGGLE_COMPETITION = ""                               # e.g. "titanic"  (URL: kaggle.com/competitions/<এই নাম>)
KAGGLE_DATASET = ""                                   # competition না হয়ে dataset হলে: "owner/dataset-name"
LOCAL_DATA_DIR = Path("data/nlp")

def unzip_all(folder):
    while zips := list(Path(folder).rglob("*.zip")):                       # zip এর ভেতরে zip থাকলেও (Kaggle এ common)
        for z in zips:
            print("Unzipping", z.name)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(z.parent)
            z.unlink()

if IN_COLAB and DATA_SOURCE != "demo":
    from google.colab import files, drive
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    if DATA_SOURCE == "upload":
        for name in files.upload():
            shutil.move(name, LOCAL_DATA_DIR / name)
    elif DATA_SOURCE == "drive":
        drive.mount("/content/drive")
        shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR, dirs_exist_ok=True)   # local copy → Drive থেকে পড়ার চেয়ে অনেক দ্রুত
    elif DATA_SOURCE == "kaggle":
        # নতুন token (KGAT_...) বা পুরনো kaggle.json — যেটা আছে সেটা দিয়েই চলবে
        import getpass
        if not os.environ.get("KAGGLE_API_TOKEN") and not Path("~/.kaggle/kaggle.json").expanduser().exists():
            tok = getpass.getpass("Kaggle API token paste করো (লুকানো থাকবে, KGAT_... দিয়ে শুরু) অথবা খালি রেখে Enter দিয়ে kaggle.json upload করো: ").strip()
            if tok:
                os.environ["KAGGLE_API_TOKEN"] = tok
            else:
                files.upload()
                Path("~/.kaggle").expanduser().mkdir(exist_ok=True)
                shutil.move("kaggle.json", Path("~/.kaggle/kaggle.json").expanduser())
                os.chmod(Path("~/.kaggle/kaggle.json").expanduser(), 0o600)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=False)
        if KAGGLE_COMPETITION:
            subprocess.run(["kaggle", "competitions", "download", "-c", KAGGLE_COMPETITION, "-p", str(LOCAL_DATA_DIR)], check=True)
        else:
            subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(LOCAL_DATA_DIR)], check=True)
    unzip_all(LOCAL_DATA_DIR)

if LOCAL_DATA_DIR.exists():
    print(f"\n📁 {LOCAL_DATA_DIR} এর ভেতরে:")
    for p in sorted(LOCAL_DATA_DIR.iterdir())[:30]:
        print("  ", p.name, "/" if p.is_dir() else f"({p.stat().st_size / 1e6:.2f} MB)")
print("IN_COLAB =", IN_COLAB, "| DATA_SOURCE =", DATA_SOURCE)

## 0. Setup & Config

In [ ]:
import os, re, random, time, string, importlib, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn import metrics

def try_import(name):
    try:
        return importlib.import_module(name)
    except ImportError:
        print(f"⚠️  '{name}' নেই → pip install {name}")
        return None

lgb = try_import("lightgbm")
nltk = try_import("nltk")
wordcloud = try_import("wordcloud")
torch = try_import("torch")
transformers = try_import("transformers")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 200)

In [ ]:
# ======================= CONFIG =======================
DATA_DIR = Path("data/nlp")
TRAIN_FILE = DATA_DIR / "train.csv"
TEST_FILE = DATA_DIR / "test.csv"
SUBMISSION_FILE = DATA_DIR / "sample_submission.csv"
OUTPUT_DIR = Path("outputs/nlp")

TEXT_COL = "text"
TARGET = "label"
ID_COL = "id"
LANG = "en"                 # "en" / "bn" (বাংলা) — preprocessing এ পার্থক্য আছে
METRIC = "f1_macro"         # "accuracy", "f1" (binary), "f1_macro", "f1_weighted", "roc_auc", "neg_log_loss"
                            # roc_auc / neg_log_loss হলে submission এ probability যাবে
SEED = 42
N_FOLDS = 5
N_JOBS = 2                  # parallel CPU worker; RAM কম হলে 1, অনেক RAM হলে -1 (সব core)

# Transformer
RUN_TRANSFORMER = bool(torch is not None and torch.cuda.is_available())   # GPU থাকলে auto ON; CPU তে জোর করে True দিলে slow
TRANSFORMER_MODEL = "distilbert-base-uncased"
# English: "microsoft/deberta-v3-small", "roberta-base" | Multilingual: "xlm-roberta-base"
# বাংলা: "csebuetnlp/banglabert", "sagorsarker/bangla-bert-base"
MAX_LEN = 128
TR_EPOCHS = 3
TR_BATCH = 16
TR_LR = 2e-5
TR_FOLDS = [0]
# ======================================================

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    if torch is not None:
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### (Practice) Demo dataset generator

In [ ]:
def make_demo_text(n=3000, n_test=750, seed=SEED):
    rng = np.random.default_rng(seed)
    P = ["I love it", "excellent quality", "absolutely amazing", "highly recommend", "super fast delivery",
         "works perfectly", "very happy with this", "worth every penny", "best purchase ever", "awesome product"]
    N = ["terrible experience", "broke after two days", "waste of money", "very disappointed", "poor quality",
         "delivery was late", "does not work", "I want a refund", "worst product", "never buying again"]
    U = ["it is okay", "average product", "as described", "nothing special", "arrived on time",
         "it is fine I guess", "decent for the price", "pretty normal", "what I expected", "not bad not good"]
    prods = ["phone", "headphones", "laptop", "shoes", "watch", "charger", "book", "t-shirt", "camera", "bag"]
    extras = ["", "", "", " Check http://shop.example.com/item{}", " <br/>", " #deal", " @seller", " 😍", " 😡", " 🙂", "!!!"]
    pools = {"positive": (P, [4, 5]), "negative": (N, [1, 2]), "neutral": (U, [3, 3])}

    def one(label):
        main, stars = pools[label]
        other = [k for k in pools if k != label]
        phrases = list(rng.choice(main, 2, replace=False))
        if rng.random() < 0.3:                                  # confusing phrase from another class
            phrases.append(str(rng.choice(pools[str(rng.choice(other))][0])))
        rng.shuffle(phrases)
        s = f"The {rng.choice(prods)}: " + ". ".join(phrases) + "."
        if rng.random() < 0.5:
            s += f" {int(rng.integers(stars[0], stars[1] + 1))} stars."
        s += str(rng.choice(extras)).format(int(rng.integers(100, 999)))
        return s.upper() if rng.random() < 0.05 else s

    labels = rng.choice(list(pools), n + n_test, p=[.5, .3, .2])
    texts = [one(str(l)) for l in labels]
    flip = rng.random(len(labels)) < 0.05                       # 5% label noise
    labels = np.where(flip, rng.choice(list(pools), len(labels)), labels)
    df = pd.DataFrame({"id": np.arange(len(labels)), "text": texts, "label": labels})
    tr, te = df.iloc[:n].copy(), df.iloc[n:].copy()
    tr.loc[tr.sample(frac=.01, random_state=seed).index, "text"] = np.nan          # missing text
    tr = pd.concat([tr, tr.sample(40, random_state=seed)]).sample(frac=1, random_state=seed)  # duplicates
    return tr, te.drop(columns="label"), pd.DataFrame({"id": te["id"], "label": "positive"})

if not TRAIN_FILE.exists():
    print("Data পাওয়া যায়নি → DEMO review dataset বানানো হচ্ছে")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _tr, _te, _sub = make_demo_text()
    _tr.to_csv(TRAIN_FILE, index=False); _te.to_csv(TEST_FILE, index=False); _sub.to_csv(SUBMISSION_FILE, index=False)

## 1. Load data

In [ ]:
train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE) if TEST_FILE.exists() else None
sample_sub = pd.read_csv(SUBMISSION_FILE) if SUBMISSION_FILE.exists() else None
print("train:", train.shape, "| test:", None if test is None else test.shape)
display(train.head(8))

## 2. Cleaning — missing, empty, duplicate, conflicting labels

In [ ]:
print("Missing text:", train[TEXT_COL].isna().sum(), "| Missing label:", train[TARGET].isna().sum())
train = train.dropna(subset=[TARGET]).copy()
train[TEXT_COL] = train[TEXT_COL].fillna("").astype(str)
if test is not None:
    test[TEXT_COL] = test[TEXT_COL].fillna("").astype(str)
print("Empty text after strip:", (train[TEXT_COL].str.strip() == "").sum())
train = train[train[TEXT_COL].str.strip() != ""]

dup_mask = train.duplicated(subset=[TEXT_COL, TARGET])
conflict = train.groupby(TEXT_COL)[TARGET].nunique()
print(f"Exact duplicates: {dup_mask.sum()} | Same text, different label: {(conflict > 1).sum()}")
train = train[~dup_mask].reset_index(drop=True)
# Conflicting হলে majority label রাখা যায়:
# train = train.groupby(TEXT_COL, as_index=False)[TARGET].agg(lambda s: s.mode()[0])

if test is not None:
    leak = set(train[TEXT_COL]) & set(test[TEXT_COL])
    print(f"Train-test overlapping texts: {len(leak)}  (থাকলে সরাসরি train label ব্যবহার করা যায়!)")

LE = LabelEncoder().fit(train[TARGET])
y = pd.Series(LE.transform(train[TARGET]), name="y")
CLASSES = [str(c) for c in LE.classes_]
N_CLASSES = len(CLASSES)
print("Classes:", CLASSES, "| train size:", len(train))

## 3. EDA
### 3.1 Class balance + text length

In [ ]:
train["n_chars"] = train[TEXT_COL].str.len()
train["n_words"] = train[TEXT_COL].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
vc = train[TARGET].value_counts()
sns.barplot(x=vc.index.astype(str), y=vc.values, ax=axes[0]); axes[0].set_title(f"Class balance (ratio {vc.max() / vc.min():.2f})")
sns.histplot(data=train, x="n_words", hue=TARGET, bins=40, element="step", ax=axes[1]); axes[1].set_title("Word count by class")
sns.boxplot(data=train, x=TARGET, y="n_chars", ax=axes[2]); axes[2].set_title("Char length by class")
plt.tight_layout(); plt.show()
display(train.groupby(TARGET)[["n_chars", "n_words"]].describe().round(1))
print(f"95th percentile words = {train.n_words.quantile(.95):.0f}  → transformer MAX_LEN ঠিক করতে কাজে লাগে")

### 3.2 Special tokens — URL, mention, hashtag, emoji, number, uppercase

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_RE = re.compile(r"<[^>]+>")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
NUM_RE = re.compile(r"\d+")
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F000-\U0001F2FF]+")

special = pd.DataFrame({
    "url": train[TEXT_COL].str.count(URL_RE.pattern),
    "html": train[TEXT_COL].str.count(HTML_RE.pattern),
    "mention": train[TEXT_COL].str.count(MENTION_RE.pattern),
    "hashtag": train[TEXT_COL].str.count(r"#\w+"),
    "emoji": train[TEXT_COL].str.count(EMOJI_RE.pattern),
    "number": train[TEXT_COL].str.count(NUM_RE.pattern),
    "exclaim": train[TEXT_COL].str.count("!"),
    "upper_ratio": train[TEXT_COL].apply(lambda t: sum(c.isupper() for c in t) / max(1, len(t))),
})
special[TARGET] = train[TARGET]
print("% of texts containing:"); display((special.drop(columns=[TARGET, "upper_ratio"]) > 0).mean().mul(100).round(2).to_frame("%"))
display(special.groupby(TARGET).mean().round(3))      # কোনো token কি class বোঝায়?

### 3.3 Top n-grams per class

In [ ]:
def top_ngrams(texts, n=1, k=15, stop="english"):
    cv = CountVectorizer(ngram_range=(n, n), stop_words=stop if LANG == "en" else None, min_df=2)
    X = cv.fit_transform(texts)
    freq = np.asarray(X.sum(0)).ravel()
    return pd.Series(freq, cv.get_feature_names_out()).sort_values(ascending=False).head(k)

for n in [1, 2]:
    fig, axes = plt.subplots(1, N_CLASSES, figsize=(5.5 * N_CLASSES, 4.5))
    for ax, c in zip(np.atleast_1d(axes), CLASSES):
        top_ngrams(train.loc[train[TARGET].astype(str) == c, TEXT_COL], n=n).sort_values().plot.barh(ax=ax, color=sns.color_palette()[CLASSES.index(c) % 10])
        ax.set_title(f"Top {'uni' if n == 1 else 'bi'}grams — {c}")
    plt.tight_layout(); plt.show()

In [ ]:
if wordcloud is not None:
    fig, axes = plt.subplots(1, N_CLASSES, figsize=(6 * N_CLASSES, 4))
    for ax, c in zip(np.atleast_1d(axes), CLASSES):
        txt = " ".join(train.loc[train[TARGET].astype(str) == c, TEXT_COL])
        # বাংলা হলে: WordCloud(font_path="path/to/NotoSansBengali.ttf", regexp=r"[ঀ-৿]+")
        wc = wordcloud.WordCloud(width=600, height=350, background_color="white", stopwords=set(ENGLISH_STOP_WORDS),
                                 colormap="viridis", random_state=SEED).generate(txt)
        ax.imshow(wc); ax.axis("off"); ax.set_title(c)
    plt.tight_layout(); plt.show()

In [ ]:
# Train vs Test length distribution + vocabulary coverage
if test is not None:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    sns.kdeplot(train.n_words, label="train", fill=True, ax=ax); sns.kdeplot(test[TEXT_COL].str.split().str.len(), label="test", fill=True, ax=ax)
    ax.legend(); ax.set_title("Word count: train vs test"); plt.show()
    tr_vocab = set(" ".join(train[TEXT_COL]).lower().split()); te_vocab = set(" ".join(test[TEXT_COL]).lower().split())
    print(f"Train vocab {len(tr_vocab):,} | Test vocab {len(te_vocab):,} | Test words unseen in train: {len(te_vocab - tr_vocab) / len(te_vocab):.1%}")

## 4. Text Preprocessing
- **Classical model (TF-IDF):** বেশি cleaning উপকারী (lowercase, punctuation, stopword, stemming)
- **Transformer (BERT):** **কম cleaning** — শুধু URL/HTML সরাও; stopword/punctuation রেখে দাও (context লাগে)
- **Sentiment task:** emoji, "!", "not" এর মতো negation word **রাখো** — এগুলো signal!
- **বাংলা:** lowercase নেই, `।` দাড়ি punctuation, `\w` regex বাংলা কার/ফলা (া ি ু ...) কে letter ধরে না — তাই Unicode range `ঀ-৿` রাখতে হয়

In [ ]:
BN_STOP = set("""অতএব অথচ অথবা অনুযায়ী অনেক অনেকে অর্থাৎ আছে আছেন আজ আবার আমরা আমাকে আমার আমি আর আরও এ এই এক একটি
এখন এখানে এটা এটি এবং এর এরা ও ওই ওর ওরা কি কিছু কিন্তু কে কেন কোন গিয়ে গেছে চেয়ে জন্য তবে তা তাই তাকে তার তারা তিনি
তুমি তো থেকে দিয়ে দিয়ে না নিয়ে পর পরে বা বলে যা যে যদি সব সঙ্গে সাথে সে হয় হয়ে হয়েছে হবে হলে হলো""".split())
NEGATIONS = {"not", "no", "nor", "never", "don't", "doesn't", "isn't", "wasn't", "won't", "can't", "cannot", "না", "নয়", "নেই"}
EN_STOP = set(ENGLISH_STOP_WORDS) - NEGATIONS

stemmer = None
if nltk is not None:
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()                                # download লাগে না
lemmatizer = None
if nltk is not None:
    try:
        nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
        from nltk.stem import WordNetLemmatizer
        lemmatizer = WordNetLemmatizer(); lemmatizer.lemmatize("cars")
    except Exception as e:
        lemmatizer = None; print("WordNet lemmatizer unavailable (offline?) → stemming ব্যবহার করো")

def clean_text(t, lower=True, rm_url=True, rm_html=True, rm_mention=True, hashtag_to_word=True,
               rm_emoji=False, rm_num=False, rm_punct=True, rm_stop=False, stem=False, lemma=False, lang=LANG):
    t = str(t)
    if rm_html:  t = HTML_RE.sub(" ", t)
    if rm_url:   t = URL_RE.sub(" ", t); t = EMAIL_RE.sub(" ", t)
    if rm_mention: t = MENTION_RE.sub(" ", t)
    if hashtag_to_word: t = HASHTAG_RE.sub(r"\1", t)
    if rm_emoji: t = EMOJI_RE.sub(" ", t)
    if lower and lang == "en": t = t.lower()
    if rm_num:   t = NUM_RE.sub(" ", t) if lang == "en" else re.sub(r"[\d০-৯]+", " ", t)
    if rm_punct:
        keep = r"\w\s" + ("ঀ-৿" if lang == "bn" else "") + ("\U0001F300-\U0001FAFF☀-➿" if not rm_emoji else "")
        t = re.sub(rf"[^{keep}]", " ", t).replace("_", " ")
        if lang == "bn": t = t.replace("।", " ")
    toks = t.split()
    if rm_stop: toks = [w for w in toks if w not in (EN_STOP if lang == "en" else BN_STOP)]
    if lemma and lemmatizer and lang == "en": toks = [lemmatizer.lemmatize(w) for w in toks]
    elif stem and stemmer and lang == "en": toks = [stemmer.stem(w) for w in toks]
    return " ".join(toks)

ex = train[TEXT_COL].iloc[0]
print("RAW        :", ex)
print("CLASSICAL  :", clean_text(ex, rm_stop=True, stem=True))
print("TRANSFORMER:", clean_text(ex, lower=False, rm_punct=False, rm_mention=False))
print("BN example :", clean_text("আমি এই ফোনটা কিনে খুবই খুশি! দাম ১২০০০ টাকা। http://x.com", lang="bn", rm_stop=True))

In [ ]:
train["clean"] = train[TEXT_COL].apply(lambda t: clean_text(t, rm_stop=False, stem=False))
train["clean_stem"] = train[TEXT_COL].apply(lambda t: clean_text(t, rm_stop=True, stem=True))
if test is not None:
    test["clean"] = test[TEXT_COL].apply(lambda t: clean_text(t, rm_stop=False, stem=False))
    test["clean_stem"] = test[TEXT_COL].apply(lambda t: clean_text(t, rm_stop=True, stem=True))
display(train[[TEXT_COL, "clean", "clean_stem"]].head())

## 5. Meta features (hand-crafted)

In [ ]:
def meta_features(s):
    return pd.DataFrame({
        "n_chars": s.str.len(),
        "n_words": s.str.split().str.len(),
        "n_unique_words": s.apply(lambda t: len(set(t.lower().split()))),
        "avg_word_len": s.apply(lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0),
        "n_exclaim": s.str.count("!"), "n_question": s.str.count(r"\?"),
        "n_punct": s.apply(lambda t: sum(ch in string.punctuation for ch in t)),
        "upper_ratio": s.apply(lambda t: sum(ch.isupper() for ch in t) / max(1, len(t))),
        "n_emoji": s.str.count(EMOJI_RE.pattern), "n_url": s.str.count(URL_RE.pattern),
        "n_digits": s.str.count(r"\d"),
    })

META_COLS = list(meta_features(train[TEXT_COL].head()).columns)
train[META_COLS] = meta_features(train[TEXT_COL])
if test is not None:
    test[META_COLS] = meta_features(test[TEXT_COL])

plt.figure(figsize=(10, 4))
sns.heatmap(train[META_COLS].assign(y=y).corr()[["y"]].T, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Meta feature correlation with (encoded) target"); plt.show()

## 6. Vectorization
| Method | কখন |
|---|---|
| **CountVectorizer (BoW)** | NB model, simple baseline |
| **TF-IDF word (1-2 gram)** | সবচেয়ে common strong baseline |
| **TF-IDF char_wb (2-5 gram)** | typo, বানান ভুল, বাংলা/মিশ্র ভাষা, ছোট text — খুব robust |
| **Word + Char একসাথে (FeatureUnion)** | প্রায়ই best classical |
| **Embeddings (sentence-transformers)** | semantic meaning, ছোট data |

In [ ]:
def tfidf_word(max_features=100_000):
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True, max_features=max_features,
                           token_pattern=r"(?u)\b\w+\b" if LANG == "en" else r"[^\s]+")
def tfidf_char(max_features=200_000):
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=max_features)
def word_char():
    return FeatureUnion([("w", tfidf_word()), ("c", tfidf_char())])

TEXT_IN = "clean"           # কোন version এর text model এ যাবে ("clean" / "clean_stem")
Xtxt = train[TEXT_IN]
Xtxt_test = test[TEXT_IN] if test is not None else None

v = tfidf_word().fit(Xtxt)
print("Word TF-IDF vocab:", len(v.vocabulary_))
idf = pd.Series(v.idf_, v.get_feature_names_out()).sort_values()
print("Most common (low idf):", idf.head(10).index.tolist())
print("Rarest (high idf):", idf.tail(10).index.tolist())

## 7. Model Zoo (TF-IDF + classical models)

In [ ]:
CV = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)

def models_zoo():
    m = {
        "Dummy": (DummyClassifier(), tfidf_word),
        "MultinomialNB": (MultinomialNB(alpha=0.1), tfidf_word),
        "ComplementNB": (ComplementNB(alpha=0.3), tfidf_word),
        "LogReg (word)": (LogisticRegression(C=5, max_iter=3000), tfidf_word),
        "LogReg (char)": (LogisticRegression(C=5, max_iter=3000), tfidf_char),
        "LogReg (word+char)": (LogisticRegression(C=5, max_iter=3000), word_char),
        "LinearSVC (word+char)": (LinearSVC(C=0.5), word_char),
        "SGD (modified_huber)": (SGDClassifier(loss="modified_huber", alpha=1e-5, random_state=SEED), word_char),
        "Ridge": (RidgeClassifier(alpha=1.0), word_char),
        "PassiveAggressive": (PassiveAggressiveClassifier(C=0.1, random_state=SEED), word_char),
    }
    if lgb is not None:
        m["LightGBM (word)"] = (lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=31, verbose=-1, random_state=SEED),
                                lambda: tfidf_word(max_features=20_000))
    return m

results = []
for name, (clf, vec_fn) in models_zoo().items():
    t0 = time.time()
    pipe = Pipeline([("vec", vec_fn()), ("clf", clf)])
    s = cross_validate(pipe, Xtxt, y, cv=CV, scoring=METRIC, n_jobs=N_JOBS)["test_score"]
    results.append({"model": name, "cv_mean": s.mean(), "cv_std": s.std(), "time_s": time.time() - t0})
    print(f"{name:24s} {METRIC} = {s.mean():.4f} ± {s.std():.4f}  ({time.time() - t0:.1f}s)")

leaderboard = pd.DataFrame(results).sort_values("cv_mean", ascending=False).reset_index(drop=True)
display(leaderboard)
plt.figure(figsize=(9, .4 * len(leaderboard) + 1))
plt.barh(leaderboard.model[::-1], leaderboard.cv_mean[::-1], xerr=leaderboard.cv_std[::-1], color="slateblue")
plt.title(f"{N_FOLDS}-fold CV {METRIC}"); plt.show()

In [ ]:
# TF-IDF + meta features একসাথে (ColumnTransformer)
meta_pipe = Pipeline([
    ("ct", ColumnTransformer([("txt", word_char(), TEXT_IN), ("meta", StandardScaler(), META_COLS)])),
    ("clf", LogisticRegression(C=5, max_iter=3000)),
])
s = cross_validate(meta_pipe, train[[TEXT_IN] + META_COLS], y, cv=CV, scoring=METRIC, n_jobs=N_JOBS)["test_score"]
print(f"LogReg (word+char+meta) {METRIC} = {s.mean():.4f} ± {s.std():.4f}")

## 8. Hyperparameter tuning (GridSearchCV)

In [ ]:
grid = GridSearchCV(
    Pipeline([("vec", tfidf_word()), ("clf", LogisticRegression(max_iter=3000))]),
    {"vec__ngram_range": [(1, 1), (1, 2), (1, 3)], "vec__min_df": [1, 2], "clf__C": [1, 5, 20]},
    cv=CV, scoring=METRIC, n_jobs=N_JOBS)
grid.fit(Xtxt, y)
print(f"Best {METRIC}: {grid.best_score_:.4f} | {grid.best_params_}")
gr = pd.DataFrame(grid.cv_results_)
display(gr.sort_values("rank_test_score")[["params", "mean_test_score", "std_test_score"]].head(8))

## 9. OOF predictions + Evaluation + Error analysis

In [ ]:
def score_oof(y_true, prob):
    lab = prob.argmax(1)
    if METRIC == "accuracy": return metrics.accuracy_score(y_true, lab)
    if METRIC.startswith("f1"): return metrics.f1_score(y_true, lab, average=METRIC.split("_")[1] if "_" in METRIC else "binary")
    if METRIC.startswith("roc_auc"): return metrics.roc_auc_score(y_true, prob if N_CLASSES > 2 else prob[:, 1], multi_class="ovr")
    if METRIC == "neg_log_loss": return -metrics.log_loss(y_true, np.clip(prob, 1e-15, 1), labels=range(N_CLASSES))
    raise ValueError(METRIC)

FINAL_MODELS = {
    "logreg_wc": Pipeline([("vec", word_char()), ("clf", LogisticRegression(C=5, max_iter=3000))]),
    "svc_wc": Pipeline([("vec", word_char()), ("clf", CalibratedClassifierCV(LinearSVC(C=0.5), cv=3))]),   # SVC → probability
    "nb_word": Pipeline([("vec", tfidf_word()), ("clf", ComplementNB(alpha=0.3))]),
    "logreg_tuned": grid.best_estimator_,
}

OOF, TEST_PROBS = {}, {}
for name, pipe in FINAL_MODELS.items():
    OOF[name] = cross_val_predict(pipe, Xtxt, y, cv=CV, method="predict_proba", n_jobs=N_JOBS)
    if Xtxt_test is not None:
        TEST_PROBS[name] = clone(pipe).fit(Xtxt, y).predict_proba(Xtxt_test)
    print(f"{name:14s} OOF {METRIC} = {score_oof(y, OOF[name]):.4f}")

oof_ens = np.mean(list(OOF.values()), axis=0)
print(f"{'AVG ensemble':14s} OOF {METRIC} = {score_oof(y, oof_ens):.4f}")

In [ ]:
best_name = max(OOF, key=lambda k: score_oof(y, OOF[k]))
prob = OOF[best_name]; pred = prob.argmax(1)
print("Best single:", best_name)
print(metrics.classification_report(y, pred, target_names=CLASSES, digits=4))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
metrics.ConfusionMatrixDisplay.from_predictions(y, pred, display_labels=CLASSES, ax=axes[0], cmap="Blues", colorbar=False)
metrics.ConfusionMatrixDisplay.from_predictions(y, pred, display_labels=CLASSES, ax=axes[1], cmap="Greens", normalize="true", colorbar=False, values_format=".2f")
plt.tight_layout(); plt.show()

# Error analysis — সবচেয়ে confident ভুল গুলো পড়ে দেখো (label noise? sarcasm? negation?)
errs = train.assign(true=LE.inverse_transform(y), pred=LE.inverse_transform(pred), conf=prob.max(1))
errs = errs[errs.true != errs.pred].sort_values("conf", ascending=False)
print(f"Errors: {len(errs)} / {len(train)}")
display(errs[[TEXT_COL, "true", "pred", "conf"]].head(15))

## 10. Interpretation — প্রতিটা class এর সবচেয়ে important word

In [ ]:
lr_pipe = Pipeline([("vec", tfidf_word()), ("clf", LogisticRegression(C=5, max_iter=3000))]).fit(Xtxt, y)
fn = lr_pipe.named_steps["vec"].get_feature_names_out()
coef = lr_pipe.named_steps["clf"].coef_
coef_rows = coef if N_CLASSES > 2 else np.vstack([-coef[0], coef[0]])
fig, axes = plt.subplots(1, N_CLASSES, figsize=(5.5 * N_CLASSES, 5))
for ax, c, w in zip(np.atleast_1d(axes), CLASSES, coef_rows):
    top = pd.Series(w, fn).sort_values()
    pd.concat([top.head(8), top.tail(12)]).plot.barh(ax=ax, color=["tomato"] * 8 + ["seagreen"] * 12)
    ax.set_title(f"'{c}' — top +/- words")
plt.tight_layout(); plt.show()

## 11. Embeddings
### 11.1 LSA (TruncatedSVD on TF-IDF) — 2D visualization

In [ ]:
Xw = tfidf_word().fit_transform(Xtxt)
svd = TruncatedSVD(100, random_state=SEED).fit(Xw)
Z = svd.transform(Xw)
print(f"SVD-100 explained variance: {svd.explained_variance_ratio_.sum():.2%}")
idx = np.random.RandomState(SEED).choice(len(Z), min(2000, len(Z)), replace=False)
Z2 = TSNE(2, random_state=SEED, init="pca").fit_transform(Z[idx])
plt.figure(figsize=(7, 5.5))
sns.scatterplot(x=Z2[:, 0], y=Z2[:, 1], hue=LE.inverse_transform(y.iloc[idx]), s=10, alpha=.7)
plt.title("t-SNE of LSA embeddings"); plt.show()

### 11.2 Sentence-Transformers embeddings + LogisticRegression
`pip install sentence-transformers` | English: `all-MiniLM-L6-v2` | Multilingual/বাংলা: `paraphrase-multilingual-MiniLM-L12-v2`, `sentence-transformers/LaBSE`

In [ ]:
st = try_import("sentence_transformers")
if st is not None:
    try:
        st_model = st.SentenceTransformer("all-MiniLM-L6-v2" if LANG == "en" else "paraphrase-multilingual-MiniLM-L12-v2")
        E = st_model.encode(train[TEXT_COL].tolist(), batch_size=64, show_progress_bar=False, normalize_embeddings=True)
        s = cross_validate(LogisticRegression(C=2, max_iter=3000), E, y, cv=CV, scoring=METRIC)["test_score"]
        print(f"SentenceTransformer + LogReg {METRIC} = {s.mean():.4f} ± {s.std():.4f}")
        OOF["st_logreg"] = cross_val_predict(LogisticRegression(C=2, max_iter=3000), E, y, cv=CV, method="predict_proba")
        if test is not None:
            TEST_PROBS["st_logreg"] = LogisticRegression(C=2, max_iter=3000).fit(E, y).predict_proba(
                st_model.encode(test[TEXT_COL].tolist(), batch_size=64, show_progress_bar=False, normalize_embeddings=True))
    except Exception as e:
        print("Sentence-transformer skipped:", type(e).__name__, e)

## 12. Transformer Fine-tuning (BERT family) — GPU recommended
Custom PyTorch loop (Trainer API version বদলালেও এটা চলবে)।
- Text এ **light cleaning** (URL/HTML বাদ), stopword রাখো
- `MAX_LEN` = token length এর ~95th percentile
- LR 1e-5 ~ 5e-5, epoch 2-5, warmup 10%

In [ ]:
if RUN_TRANSFORMER and transformers is not None and torch is not None:
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_scheduler

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    USE_AMP = DEVICE.type == "cuda"
    tok = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)
    light = lambda s: s.apply(lambda t: clean_text(t, lower=False, rm_punct=False, rm_mention=False))
    tr_text = light(train[TEXT_COL]).tolist()
    te_text = light(test[TEXT_COL]).tolist() if test is not None else None

    tok_lens = [len(tok(t)["input_ids"]) for t in tr_text[:2000]]
    print(f"Token length p50={np.percentile(tok_lens, 50):.0f} p95={np.percentile(tok_lens, 95):.0f} max={max(tok_lens)} | MAX_LEN={MAX_LEN}")

    class TextDS(Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tok(texts, truncation=True, max_length=MAX_LEN)
            self.labels = labels
        def __len__(self):
            return len(self.enc["input_ids"])
        def __getitem__(self, i):
            item = {k: v[i] for k, v in self.enc.items()}
            if self.labels is not None:
                item["labels"] = int(self.labels[i])
            return item

    collate = DataCollatorWithPadding(tok)

    @torch.no_grad()
    def tr_predict(model, loader):
        model.eval(); out = []
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items() if k != "labels"}
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                out.append(model(**b).logits.float().softmax(-1).cpu())
        return torch.cat(out).numpy()

    tr_oof = np.full((len(train), N_CLASSES), np.nan)
    tr_test = np.zeros((len(test), N_CLASSES)) if test is not None else None
    for fold, (tr_idx, va_idx) in enumerate(CV.split(tr_text, y)):
        if fold not in TR_FOLDS:
            continue
        seed_everything(SEED + fold)
        model = AutoModelForSequenceClassification.from_pretrained(TRANSFORMER_MODEL, num_labels=N_CLASSES).to(DEVICE)
        tl = DataLoader(TextDS([tr_text[i] for i in tr_idx], y.values[tr_idx]), batch_size=TR_BATCH, shuffle=True, collate_fn=collate)
        vl = DataLoader(TextDS([tr_text[i] for i in va_idx]), batch_size=TR_BATCH * 2, collate_fn=collate)
        opt = torch.optim.AdamW(model.parameters(), lr=TR_LR, weight_decay=0.01)
        steps = TR_EPOCHS * len(tl)
        sch = get_scheduler("linear", opt, num_warmup_steps=int(.1 * steps), num_training_steps=steps)
        scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
        best, best_prob = -1, None
        for ep in range(TR_EPOCHS):
            model.train(); t0 = time.time(); tot = 0
            for b in tl:
                b = {k: v.to(DEVICE) for k, v in b.items()}
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    loss = model(**b).loss
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
            p = tr_predict(model, vl); sc = score_oof(y.values[va_idx], p)
            print(f"fold {fold} ep {ep + 1} | train loss {tot / len(tl):.4f} | val {METRIC} {sc:.4f} | {time.time() - t0:.0f}s")
            if sc > best:
                best, best_prob = sc, p
                torch.save(model.state_dict(), OUTPUT_DIR / f"transformer_fold{fold}.pt")
        tr_oof[va_idx] = best_prob
        if test is not None:
            model.load_state_dict(torch.load(OUTPUT_DIR / f"transformer_fold{fold}.pt", map_location=DEVICE))
            tr_test += tr_predict(model, DataLoader(TextDS(te_text), batch_size=TR_BATCH * 2, collate_fn=collate)) / len(TR_FOLDS)
        del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None

    m = ~np.isnan(tr_oof).any(1)
    TRANSFORMER_SCORE = score_oof(y.values[m], tr_oof[m])
    classical_same_rows = score_oof(y.values[m], OOF["logreg_wc"][m])
    print(f"Transformer OOF {METRIC} (folds {TR_FOLDS}) = {TRANSFORMER_SCORE:.4f}  vs  TF-IDF LogReg on same rows = {classical_same_rows:.4f}")
    if len(TR_FOLDS) == N_FOLDS:            # সব fold থাকলে OOF সহ ensemble এ যোগ করো
        OOF["transformer"] = tr_oof
        if test is not None: TEST_PROBS["transformer"] = tr_test
    elif test is not None and TRANSFORMER_SCORE >= classical_same_rows:
        TEST_PROBS["transformer"] = tr_test   # partial fold: classical এর চেয়ে ভালো হলেই শুধু নাও
    else:
        print("Transformer classical model এর চেয়ে খারাপ → ensemble এ নেওয়া হলো না (EPOCHS/LR/model বদলে দেখো)")
else:
    print("Transformer section skipped (RUN_TRANSFORMER=False বা library নেই)। GPU তে RUN_TRANSFORMER=True দাও।")

## 13. Ensemble + Submission

In [ ]:
# Weighted ensemble — শুধু যে model গুলোর OOF আছে সেগুলো দিয়ে weight; বাকিরা (partial-fold transformer) equal share
names = list(OOF)
scores = {n: score_oof(y, OOF[n]) for n in names}
w = np.array([scores[n] for n in names]); w = w / w.sum()
oof_final = sum(wi * OOF[n] for wi, n in zip(w, names))
print(f"Weighted ensemble OOF {METRIC} = {score_oof(y, oof_final):.4f}  | weights: {dict(zip(names, w.round(3)))}")

if test is not None:
    test_names = [n for n in TEST_PROBS]
    tw = np.array([scores.get(n, globals().get("TRANSFORMER_SCORE", 0)) for n in test_names]); tw = tw / tw.sum()
    test_final = sum(wi * TEST_PROBS[n] for wi, n in zip(tw, test_names))
    if METRIC in {"neg_log_loss", "roc_auc_ovr"}:                 # probability submission: এক class এক column
        sub = pd.DataFrame(test_final, columns=CLASSES)
        sub.insert(0, ID_COL, test[ID_COL].values)
        if sample_sub is not None and set(CLASSES) <= set(sample_sub.columns):
            sub = sub[list(sample_sub.columns)]
    elif METRIC == "roc_auc" and N_CLASSES == 2:
        sub = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_final[:, 1]})
    else:
        sub = pd.DataFrame({ID_COL: test[ID_COL], TARGET: LE.inverse_transform(test_final.argmax(1))})
    if sample_sub is not None:
        if sub.shape[1] == 2: sub.columns = sample_sub.columns[:2]
        assert len(sub) == len(sample_sub)
    sub.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    np.save(OUTPUT_DIR / "test_probs.npy", test_final)
    print("Saved →", OUTPUT_DIR / "submission.csv")
    display(sub.head()); print(sub.iloc[:, 1].value_counts(normalize=True).round(3))

## ☁️ Colab: Submission download + Drive এ backup

In [ ]:
SAVE_TO_DRIVE = False                                  # True → outputs folder Google Drive এ copy হবে
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/datathon/outputs/nlp"

if IN_COLAB:
    from google.colab import files, drive
    sub_path = OUTPUT_DIR / "submission.csv"
    if sub_path.exists():
        files.download(str(sub_path))                  # browser এ download হবে
    if SAVE_TO_DRIVE:
        drive.mount("/content/drive")
        shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
        print("Saved to Drive →", DRIVE_OUTPUT_DIR)
else:
    print("Local run — files আছে:", OUTPUT_DIR.resolve())

## 14. অন্য NLP task হলে কি বদলাবো?

| Task | পরিবর্তন |
|---|---|
| **Multi-label** (একটা text এ একাধিক tag) | `MultiLabelBinarizer` + `OneVsRestClassifier(LogisticRegression())`; transformer এ `problem_type="multi_label_classification"` + sigmoid + threshold tune |
| **Text regression** (rating/score) | `Ridge` / `LGBMRegressor` on TF-IDF; transformer এ `num_labels=1` (MSE loss auto) |
| **NER / token classification** | `AutoModelForTokenClassification`, label align with `word_ids()` |
| **Text pair** (similarity, NLI, QA match) | `tok(text1, text2, ...)` pair encoding; বা sentence embedding cosine similarity |
| **খুব কম labeled data** | Sentence embedding + LogReg, বা zero-shot (`pipeline("zero-shot-classification")`) |
| **বাংলা / Banglish** | char n-gram TF-IDF (খুব কাজের), `csebuetnlp/banglabert`, `xlm-roberta-base`; Unicode normalize: `unicodedata.normalize("NFC", t)` |
| **Long documents** | head+tail truncation (প্রথম 128 + শেষ 382 token), অথবা chunk করে average |

## ✅ Score বাড়ানোর checklist
1. TF-IDF word + char একসাথে, `sublinear_tf=True`, `C` tune
2. Error analysis পড়ে cleaning বদলাও (negation রাখো, emoji রাখো)
3. Transformer: সব fold train → OOF → TF-IDF model এর সাথে weighted ensemble
4. বড় model (deberta-v3-base/large) + বেশি MAX_LEN, কম LR
5. Pseudo-labeling: confident test prediction কে train এ যোগ করে আবার train